In [0]:
%pip install -U kaleido plotly
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")


In [0]:
import pandas as pd
from src.data.collectors.weather import _DNO_REGIONS

# Load results from both model runs
results_04 = pd.read_csv("/Volumes/workspace/default/models/04_train_regional_models.csv")
results_04 = results_04[results_04['region_id'] <= 14]
results_04['model_type'] = 'no_weather'

results_08 = pd.read_csv("/Volumes/workspace/default/models/08_train_regional_models_with_weather.csv")
results_08['model_type'] = 'with_weather'

print(f"Model 04 (without weather): {len(results_04)} rows")
print(f"Model 08 (with weather): {len(results_08)} rows")

In [ ]:
import plotly.express as px

# Filter to P50 and combine
results_04_50 = results_04[results_04['alpha'] == 0.5]
results_08_50 = results_08[results_08['alpha'] == 0.5]
alpha_50_combined = pd.concat([results_04_50, results_08_50])

# Grouped bar chart: baseline vs weather models by region
fig = px.bar(
    alpha_50_combined,
    x="region",
    y="pinball_loss",
    color="model_type",
    barmode="group",
    labels={"region": "DNO Region", "pinball_loss": "P50 Pinball Loss", "model_type": "Model Type"},
    title="P50 Pinball Loss by Region: Baseline vs Weather Features",
    color_discrete_map={"no_weather": "#636EFA", "with_weather": "#EF553B"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.write_image(f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/docs/images/compare_models.png")
fig.show()

In [ ]:
import pandas as pd

# Build lat/lon lookup from DNO regions
dno_coords = pd.DataFrame([
    {"region_id": rid, "latitude": info["latitude"], "longitude": info["longitude"]}
    for rid, info in _DNO_REGIONS.items()
])

# Join P50 weather model results with coordinates
results_08_50_combined = results_08_50.merge(dno_coords, on="region_id")

# Accuracy label: lower pinball loss = higher accuracy
results_08_50_combined["Accuracy"] = pd.cut(
    results_08_50_combined["pinball_loss"],
    bins=[0, 2.1, 5.5, float("inf")],
    labels=["LOW", "MEDIUM", "HIGH"]
)

# Geoplot: P50 pinball loss mapped across all 14 DNO regions
fig = px.scatter_mapbox(
    results_08_50_combined,
    lat="latitude",
    lon="longitude",
    color="pinball_loss",
    size="pinball_loss",
    hover_name="region",
    hover_data={"latitude": False, "longitude": False, "Accuracy": True, "pinball_loss": True},
    color_continuous_scale="Viridis",
    mapbox_style="open-street-map",
    zoom=4,
    center={"lat": 54.5, "lon": -2.5},
    title="P50 Pinball Loss by DNO Region"
)
fig.write_image(f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/docs/images/Models_Geoplot.png")
fig.show()

In [ ]:
import joblib
import re
import numpy as np

# Load gold table and generate P50 predictions per region using saved models
df_gold = spark.read.table("gold_carbon_weather_regional")
df_gold = df_gold.dropna(subset=["lag_1", "lag_2", "lag_48", "lag_336"])
gold_pdf = df_gold.toPandas()

feature_cols = ["lag_1", "lag_2", "lag_48", "lag_336", "rolling_avg_7day",
                "temperature", "wind_speed", "radiation", "temp_roll_avg_7day"]

records = []
for region_name in sorted(gold_pdf["region_name"].unique()):
    safe_region = re.sub(r"[^a-z0-9]", "_", region_name.lower())
    model_path = f"/Volumes/workspace/default/models/lgbm_p50_{safe_region}.pkl"
    try:
        model = joblib.load(model_path)
    except FileNotFoundError:
        print(f"No model for {region_name}, skipping")
        continue

    region_df = gold_pdf[gold_pdf["region_name"] == region_name].copy()
    X = region_df[feature_cols]
    y = region_df["carbon_intensity"].values
    preds = model.predict(X)

    threshold = np.median(y)
    predicted_charge = preds < threshold
    actual_optimal = y < threshold
    wrong = predicted_charge != actual_optimal

    records.append({
        "region": region_name,
        "mae_gco2": np.abs(y - preds).mean(),
        "decision_error_pct": wrong.mean() * 100,
        "carbon_penalty_gco2": np.abs(y[wrong] - threshold).mean() if wrong.any() else 0.0,
        "threshold_gco2": threshold,
    })

decision_df = pd.DataFrame(records).sort_values("decision_error_pct", ascending=False)
print(decision_df.to_string(index=False))

In [ ]:
import plotly.graph_objects as go

# Sort by decision error (worst first) for visual clarity
plot_df = decision_df.sort_values("decision_error_pct", ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    name="MAE (gCO₂/kWh)",
    y=plot_df["region"],
    x=plot_df["mae_gco2"],
    orientation="h",
    marker_color="#636EFA"
))

fig.add_trace(go.Bar(
    name="Avg carbon penalty when wrong (gCO₂/kWh)",
    y=plot_df["region"],
    x=plot_df["carbon_penalty_gco2"],
    orientation="h",
    marker_color="#EF553B"
))

fig.add_trace(go.Scatter(
    name="Decision error rate (%)",
    y=plot_df["region"],
    x=plot_df["decision_error_pct"],
    mode="markers",
    marker=dict(color="#00CC96", size=10, symbol="diamond"),
    xaxis="x2"
))

fig.update_layout(
    title="Forecast quality by region: MAE, carbon penalty, and decision error rate",
    barmode="group",
    height=600,
    xaxis=dict(title="gCO₂/kWh", domain=[0, 0.75]),
    xaxis2=dict(title="Decision error rate (%)", overlaying="x", side="top", domain=[0, 0.75]),
    yaxis=dict(title="DNO Region"),
    legend=dict(orientation="h", yanchor="bottom", y=-0.2)
)

fig.write_image(f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/docs/images/decision_quality_by_region.png")
fig.show()